# 02 — Exploratory Data Analysis on Bronze

Distribution, correlation, and delay-pattern audit on the raw table. Nothing here writes back — this notebook is documentation, not a pipeline step.

Every cell mixes Databricks' interactive `display()` (for the notebook UI) with matplotlib (which survives HTML export). Screenshot any chart with a clean rendering — those go into `docs/screenshots/`.


## Setup


In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql.functions import (
    col, count, floor, mean, month, stddev, sum as _sum,
    to_date, when, year,
)

from src import config

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titleweight": "bold",
})


In [ ]:
bronze = spark.table(config.BRONZE)
bronze_count = bronze.count()
print(f"Bronze rows: {bronze_count:,}")
print(f"Columns:     {len(bronze.columns)}")


## 1. Column-level null rates

Cancellation-related columns dominate — expected.


In [ ]:
null_report = bronze.select([
    (count(when(col(c).isNull(), c)) / bronze_count).alias(c)
    for c in bronze.columns
])
display(null_report)


## 2. Arrival-delay distribution

Clipped to [-60, 240] so the tail doesn't destroy the histogram scale. Roughly bimodal: on-time cluster around 0, delayed shoulder past +15.


In [ ]:
delay_pdf = (
    bronze.select("ARR_DELAY")
    .filter(col("ARR_DELAY").isNotNull())
    .sample(fraction=0.05, seed=42)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(
    delay_pdf["ARR_DELAY"].clip(-60, 240),
    bins=80, color="#1f77b4", edgecolor="white",
)
ax.axvline(15, ls="--", c="crimson", lw=1.5, label="FAA on-time cutoff (+15 min)")
ax.axvline(0, ls=":", c="black", lw=1, label="On-time")
ax.set_xlabel("Arrival delay (minutes)")
ax.set_ylabel("Flights (5% sample)")
ax.set_title("Distribution of arrival delay — 5% sample, clipped ±")
ax.legend()
plt.tight_layout()
plt.show()


## 3. On-time performance by year

2020's collapse in flight volume masks a real drop in delay too — Silver drops the year for that reason.


In [ ]:
by_year = (
    bronze
    .withColumn("flight_year", year(to_date(col("FL_DATE"))))
    .groupBy("flight_year")
    .agg(
        count("*").alias("flights"),
        mean("ARR_DELAY").alias("mean_arr_delay"),
        (mean(when(col("ARR_DELAY") < 15, 1.0).otherwise(0.0)) * 100).alias("on_time_pct"),
    )
    .orderBy("flight_year")
)
display(by_year)

pdf = by_year.toPandas()
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(pdf["flight_year"].astype(str), pdf["flights"] / 1e6,
        color="#c9d3e0", label="Flights (millions)")
ax1.set_ylabel("Flights (millions)", color="#5a6a80")
ax2 = ax1.twinx()
ax2.plot(pdf["flight_year"].astype(str), pdf["on_time_pct"],
         marker="o", color="#d62728", lw=2, label="On-time %")
ax2.set_ylabel("On-time % (arr_delay < 15)", color="#d62728")
ax2.grid(False)
ax1.set_title("Flight volume and on-time performance by year")
plt.tight_layout()
plt.show()


## 4. Seasonality — mean delay by month

December and June/July are worst; a shoulder period in Sept–Oct is calmest.


In [ ]:
by_month = (
    bronze
    .withColumn("m", month(to_date(col("FL_DATE"))))
    .groupBy("m")
    .agg(mean("ARR_DELAY").alias("mean_delay"),
         count("*").alias("flights"))
    .orderBy("m")
    .toPandas()
)

labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(labels, by_month["mean_delay"], color="#2ca02c")
ax.set_ylabel("Mean arrival delay (min)")
ax.set_title("Seasonality of arrival delay by month")
plt.tight_layout()
plt.show()


## 5. Day-of-week pattern

Thursday–Friday runs hottest; Saturday is the calmest by a clear margin.


In [ ]:
from pyspark.sql.functions import dayofweek

by_dow = (
    bronze
    .withColumn("dow", dayofweek(to_date(col("FL_DATE"))))
    .groupBy("dow")
    .agg(mean("ARR_DELAY").alias("mean_delay"))
    .orderBy("dow")
    .toPandas()
)

dow_labels = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dow_labels, by_dow["mean_delay"], color="#9467bd")
ax.set_ylabel("Mean arrival delay (min)")
ax.set_title("Arrival delay by day of week")
plt.tight_layout()
plt.show()


## 6. Delay by scheduled departure hour — the afternoon-cascade curve

This is the pattern most portfolios miss. Delay accumulates through the day because late aircraft propagate downstream schedules.


In [ ]:
by_hour = (
    bronze
    .withColumn("dep_hour", floor(col("CRS_DEP_TIME") / 100))
    .filter((col("dep_hour") >= 0) & (col("dep_hour") <= 23))
    .groupBy("dep_hour")
    .agg(mean("ARR_DELAY").alias("mean_delay"),
         count("*").alias("flights"))
    .orderBy("dep_hour")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(by_hour["dep_hour"], by_hour["mean_delay"],
        marker="o", lw=2, color="#d62728")
ax.fill_between(by_hour["dep_hour"], 0, by_hour["mean_delay"], alpha=0.15, color="#d62728")
ax.axhline(15, ls="--", c="black", lw=1, alpha=0.5, label="FAA on-time cutoff")
ax.set_xlabel("Scheduled departure hour (local)")
ax.set_ylabel("Mean arrival delay (min)")
ax.set_title("Mean arrival delay by scheduled departure hour")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()


## 7. Delay by airline


In [ ]:
by_airline = (
    bronze
    .groupBy("AIRLINE")
    .agg(count("*").alias("flights"),
         mean("ARR_DELAY").alias("mean_delay"))
    .orderBy(col("mean_delay").desc())
    .toPandas()
)
display(spark.createDataFrame(by_airline))

fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(by_airline))))
ax.barh(by_airline["AIRLINE"], by_airline["mean_delay"], color="#ff7f0e")
ax.invert_yaxis()
ax.set_xlabel("Mean arrival delay (min)")
ax.set_title("Mean arrival delay by airline")
plt.tight_layout()
plt.show()


## 8. Cancellation rate by airline


In [ ]:
cancel = (
    bronze
    .groupBy("AIRLINE")
    .agg(
        count("*").alias("flights"),
        (mean(when(col("CANCELLED") == 1, 1.0).otherwise(0.0)) * 100).alias("cancel_pct"),
    )
    .orderBy(col("cancel_pct").desc())
    .toPandas()
)
display(spark.createDataFrame(cancel))

fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(cancel))))
ax.barh(cancel["AIRLINE"], cancel["cancel_pct"], color="#8c564b")
ax.invert_yaxis()
ax.set_xlabel("Cancellation rate (%)")
ax.set_title("Cancellation rate by airline")
plt.tight_layout()
plt.show()


## 9. Delay-cause breakdown

The DELAY_DUE_* columns are only populated for delayed flights, so this shows the *composition* of delay time when delay exists — not overall rates.


In [ ]:
cause_cols = [
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT",
]
totals = (
    bronze.select([_sum(col(c)).alias(c) for c in cause_cols])
    .toPandas()
    .T.reset_index()
    .rename(columns={"index": "cause", 0: "minutes"})
)
totals["share"] = totals["minutes"] / totals["minutes"].sum() * 100
totals["cause"] = totals["cause"].str.replace("DELAY_DUE_", "").str.title()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#7f7f7f", "#d62728"]
ax.barh(totals["cause"], totals["share"], color=colors)
for i, v in enumerate(totals["share"]):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
ax.set_xlabel("Share of total delay minutes (%)")
ax.set_title("Delay-cause composition (when delay exists)")
plt.tight_layout()
plt.show()


## 10. Top 20 worst routes

Restricted to routes with ≥1,000 flights so tiny-sample outliers don't dominate.


In [ ]:
from pyspark.sql.functions import concat_ws

routes = (
    bronze
    .withColumn("route", concat_ws(" → ", col("ORIGIN"), col("DEST")))
    .groupBy("route")
    .agg(count("*").alias("flights"),
         mean("ARR_DELAY").alias("mean_delay"))
    .filter(col("flights") >= 1000)
    .orderBy(col("mean_delay").desc())
    .limit(20)
    .toPandas()
)
display(spark.createDataFrame(routes))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(routes["route"], routes["mean_delay"], color="#e377c2")
ax.invert_yaxis()
ax.set_xlabel("Mean arrival delay (min)")
ax.set_title("Top 20 worst routes by mean arrival delay (≥1k flights)")
plt.tight_layout()
plt.show()


## 11. Distance vs delay — is there a distance effect?

Sampled 10k rows. The relationship is weak; length of flight is a poor delay predictor.


In [ ]:
scatter = (
    bronze
    .select("DISTANCE", "ARR_DELAY")
    .filter(col("ARR_DELAY").isNotNull())
    .sample(fraction=0.005, seed=42)
    .limit(10000)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(scatter["DISTANCE"], scatter["ARR_DELAY"].clip(-60, 240),
           alpha=0.15, s=8, color="#17becf")
ax.axhline(0, c="black", lw=0.5)
ax.axhline(15, ls="--", c="crimson", lw=1)
ax.set_xlabel("Distance (mi)")
ax.set_ylabel("Arrival delay (min, clipped)")
ax.set_title("Arrival delay vs distance (10k sample)")
plt.tight_layout()
plt.show()

corr = scatter["DISTANCE"].corr(scatter["ARR_DELAY"])
print(f"Pearson correlation:  {corr:+.3f}")


## 12. Correlation heatmap of numerical features

Confirms the intuition that `DEP_DELAY` dominates `ARR_DELAY` prediction — the whole reason we ship two models.


In [ ]:
num_cols = ["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN",
            "AIR_TIME", "DISTANCE", "CRS_ELAPSED_TIME"]
corr_pdf = (
    bronze.select(*num_cols)
    .dropna()
    .sample(0.01, seed=42)
    .toPandas()
    .corr()
)

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(corr_pdf, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr_pdf.iloc[i, j]:.2f}",
                ha="center", va="center",
                color="white" if abs(corr_pdf.iloc[i, j]) > 0.5 else "black", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation heatmap — numerical features")
plt.tight_layout()
plt.show()


## Notes for Silver

- 2020 will be dropped (COVID anomaly visible in chart 3).
- Actual times, taxi times, delay-cause breakdowns are dropped: unavailable at booking time (target leakage for the pre-departure model).
- `DEP_DELAY` is retained but only fed to the **in-flight** model — the pre-departure model must live without it.
